In [ ]:

from google.cloud import bigquery  
from google.oauth2 import service_account 

import pandas as pd  
import pandas_gbq

pd.set_option("display.max_columns", None)  
pd.set_option("display.max_rows", 10)  

In [ ]:

meu_service_account_file = r"C:\Users\1578090\Downloads\meu-proj-eng-dados-606443cfb83f.json"

meu_gcp_project_id =  "meu-proj-eng-dados"

gcp_credentials = service_account.Credentials.from_service_account_file(meu_service_account_file)

bigquery_client = bigquery.Client(credentials=gcp_credentials, project=meu_gcp_project_id)

In [ ]:

df_boxoffice_dc_marvel = bigquery_client.query("""
    SELECT *
    FROM bronze.boxoffice_dc_marvel
    """).to_dataframe(create_bqstorage_client=False)

In [4]:
df_boxoffice_dc_marvel.head(4)  

,int64_field_0,FILM,US_RELEASE_DATE,BOX_OFFICE_GROSS_DOMESTIC_US_AND_CANADA,BOX_OFFICE_GROSS_OTHER_TERRITORIES,BOX_OFFICE_GROSS_WORLDWIDE,BUDGET,MCU,PHASE,DISTRIBUTOR,MPAA_RATING,LENGTH,MINUTES,FRANCHISE,CHARACTER_FAMILY,DOMESTIC,GROSS_TO_BUDGET,ROTTEN_TOMATOES_CRITIC_SCORE,MALE_FEMALE_LED,YEAR,INFLATION_ADJUSTED_WORLDWIDE_GROSS,INFLATION_ADJUSTED_BUDGET,25X_PROD,BREAK_EVEN
0,113,NaN,NaT,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN
1,89,The New Mutants,2020-08-28,23852659,25316935,49169594,67000000,False,NaN,20th Century Studios,PG-13,01:34:00,94.0,Marvel,X-Men,0.49,0.73,36.0,Co-starring,2020.0,49169594,67000000,167500000,Flop
2,98,Morbius,2022-04-01,73865530,93595431,167460961,75000000,False,NaN,Columbia Pictures,PG-13,01:44:00,104.0,Marvel,Spider-Man Allies and Villains,0.44,2.23,15.0,Male,2022.0,167460961,75000000,187500000,Flop
3,112,Madame Web,2024-02-14,42619699,54000000,96619699,80000000,False,NaN,Columbia Pictures,PG-13,01:56:00,116.0,Marvel,Spider-Man Allies and Villains,0.44,1.21,13.0,Female,2024.0,96619699,80000000,200000000,Flop


In [ ]:

df_imdb_filmes = bigquery_client.query("SELECT * FROM bronze.imdb_filmes ").to_dataframe(create_bqstorage_client=False)

In [6]:
df_imdb_filmes.head(4)

,Unnamed: 0,ID,TITLE,LINK,YEAR,DURATION,RATING_MPA,RATING_IMDB,VOTE,BUDGET,GROSS_WORLD_WIDE,GROSS_US_CANADA,GROSS_OPENING_WEEKEND,DIRECTOR,WRITER,STAR,GENRE,COUNTRY_ORIGIN,FILMING_LOCATION,PRODUCTION_COMPANY,LANGUAGE,WIN,NOMINATION,OSCAR
0,0,tt0073470,Strip Nude for Your Killer,https://www.imdb.com/title/tt0073470,1975,1h 38m,Not Rated,5.6,33000.0,NaN,NaN,NaN,NaN,Andrea Bianchi,"Andrea Bianchi, Massimo Felisatti","Edwige Fenech, Nino Castelnuovo, Femi Benussi","Erotic Thriller, Giallo, Slasher Horror, Horro...",Italy,"Milan, Lombardia, Italy",FRAL Cinematografica,Italian,0,0,0
1,1,tt0072764,Carry on Behind,https://www.imdb.com/title/tt0072764,1975,1h 30m,Not Rated,5.6,31000.0,NaN,NaN,NaN,NaN,Gerald Thomas,Dave Freeman,"Elke Sommer, Kenneth Williams, Bernard Bresslaw","Parody, Slapstick, Comedy",United Kingdom,"Bad Godesberg Way, Maidenhead, Berkshire, Engl...","The Rank Organisation, Peter Rogers Productions",English,0,0,0
2,2,tt0131526,"So Young, So Lovely, So Vicious...",https://www.imdb.com/title/tt0131526,1975,1h 30m,NaN,4.9,487.0,NaN,NaN,NaN,NaN,Silvio Amadio,"Silvio Amadio, Roberto Natale","Gloria Guida, Dagmar Lassander, Fred Robsahm","Drama, Mystery, Thriller",Italy,"Sardinia, Italy",Domiziana Internazionale Cinematografica,Italian,0,0,0
3,3,tt0073335,Mahogany,https://www.imdb.com/title/tt0073335,1975,1h 49m,PG,6.1,23000.0,NaN,NaN,5000000.0,NaN,"Berry Gordy, Tony Richardson, Jack Wormser","John Byrum, Toni Amber","Diana Ross, Billy Dee Williams, Anthony Perkins","Drama, Romance",United States,Marshall Field Co Department Store 111 N Sta...,"Motown Productions, Nikor Productions, Paramou...","English, Italian",0,0,1


In [ ]:

df_complete_movie_info = (
    pd.merge(
        df_boxoffice_dc_marvel,
        df_imdb_filmes,
        left_on=["FILM", "YEAR"],
        right_on=["TITLE", "YEAR"],
        how="outer",
    )
    .sort_values(by="YEAR", ascending=False)
    .drop_duplicates(keep="last")
)

In [ ]:

df_complete_dc_marvel = df_complete_movie_info[
    (df_complete_movie_info["FRANCHISE"] == "Marvel") | (df_complete_movie_info["FRANCHISE"] == "DC")
]

In [ ]:

df_complete_dc_marvel = df_complete_dc_marvel.drop(
    columns=[
        "25X_PROD",
        "BOX_OFFICE_GROSS_DOMESTIC_US_AND_CANADA",
        "BOX_OFFICE_GROSS_WORLDWIDE",
        "BUDGET_y",
        "DURATION",
        "LENGTH",
        "MPAA_RATING",
        "TITLE",
        "YEAR",
    ]
).reset_index(drop=True)

In [ ]:

df_complete_dc_marvel.sort_values(by="RATING_IMDB", ascending=False).head(2)

,int64_field_0,FILM,US_RELEASE_DATE,BOX_OFFICE_GROSS_OTHER_TERRITORIES,BUDGET_x,MCU,PHASE,DISTRIBUTOR,MINUTES,FRANCHISE,CHARACTER_FAMILY,DOMESTIC,GROSS_TO_BUDGET,ROTTEN_TOMATOES_CRITIC_SCORE,MALE_FEMALE_LED,INFLATION_ADJUSTED_WORLDWIDE_GROSS,INFLATION_ADJUSTED_BUDGET,BREAK_EVEN,Unnamed: 0,ID,LINK,RATING_MPA,RATING_IMDB,VOTE,GROSS_WORLD_WIDE,GROSS_US_CANADA,GROSS_OPENING_WEEKEND,DIRECTOR,WRITER,STAR,GENRE,COUNTRY_ORIGIN,FILMING_LOCATION,PRODUCTION_COMPANY,LANGUAGE,WIN,NOMINATION,OSCAR
75,36,The Dark Knight,2008-07-18,472349861,185000000,False,NaN,Warner Bros.,152.0,DC,Batman,0.53,5.45,94.0,Male,1425605587,261816105,Success,3230,tt0468569,https://www.imdb.com/title/tt0468569,PG-13,9.0,3000000.0,1.009054e+09,534987076.0,158411483.0,Christopher Nolan,"Jonathan Nolan, Christopher Nolan, David S Goyer","Christian Bale, Heath Ledger, Aaron Eckhart","Action Epic, Epic, Superhero, Action, Crime, D...","United States, United Kingdom","Chicago, Illinois, USA","Warner Bros, Legendary Entertainment, Syncopy","English, Mandarin",0,164,0
6,107,Spider-Man: Across the Spider-Verse,2023-06-02,309205354,100000000,False,NaN,Columbia Pictures,140.0,Marvel,Spider-Man,0.55,6.91,96.0,Male,690516673,100000000,Success,25318,tt9362722,https://www.imdb.com/title/tt9362722,PG,8.5,428000.0,6.908247e+08,381593754.0,120663589.0,"Joaquim Dos Santos, Kemp Powers, Justin K Thom...","Phil Lord, Christopher Miller, Dave Callaham","Shameik Moore, Hailee Steinfeld, Brian Tyree H...","Computer Animation, Superhero, Supernatural Fa...",United States,NaN,"Columbia Pictures, Marvel Entertainment, Avi A...",English,0,164,1


In [ ]:

df_tratado = df_complete_dc_marvel.copy()

In [ ]:

percent_cols = ["DOMESTIC"]

for col in percent_cols:
    df_tratado[col] = df_tratado[col].astype(str).str.replace("%", "").astype(float)

In [ ]:

date_cols = ["US_RELEASE_DATE"]

for col in date_cols:
    df_tratado[col] = pd.to_datetime(df_tratado[col], format="%d/%m/%Y")

In [ ]:

int_cols = [
    "VOTE",
    "WIN",
    "OSCAR",
    "NOMINATION",
    "ROTTEN_TOMATOES_CRITIC_SCORE",
]

for col in int_cols:
    df_tratado[col] = df_tratado[col].fillna(0).astype(int)

In [15]:
df_tratado["YEAR"] = df_tratado["US_RELEASE_DATE"].dt.year

In [ ]:

df_tratado["GENRE"] = df_tratado["GENRE"].str.split(",").str[0].str.strip()

df_tratado["COUNTRY_ORIGIN"] = df_tratado["COUNTRY_ORIGIN"].str.split(",").str[0].str.strip()

df_tratado["LANGUAGE"] = df_tratado["LANGUAGE"].str.split(",").str[0].str.strip()

df_tratado["PRODUCTION_COMPANY"] = df_tratado["PRODUCTION_COMPANY"].str.split(",").str[0].str.strip()

df_tratado["WRITER"] = df_tratado["WRITER"].str.split(",").str[0].str.strip()

In [ ]:

df_tratado = df_tratado.drop(columns=["Unnamed: 0"], errors="ignore")

df_tratado = df_tratado[sorted(df_tratado.columns)].drop_duplicates()

In [ ]:

pandas_gbq.to_gbq(
    df_tratado,
    project_id=meu_gcp_project_id,
    destination_table="silver.complete_dc_marvel",
    if_exists="replace",
    credentials=gcp_credentials,
)